In [1]:
import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from nltk.corpus import stopwords

In [2]:
data = pd.read_csv("Resume.csv")

data.head()

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [3]:
data = data[['Resume_str','Category']]

data.head()

,Resume_str,Category
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,HR
1,"HR SPECIALIST, US HR OPERATIONS ...",HR
2,HR DIRECTOR Summary Over 2...,HR
3,HR SPECIALIST Summary Dedica...,HR
4,HR MANAGER Skill Highlights ...,HR


In [4]:
stop_words = set(stopwords.words('english'))

def clean_resume(text):

    text = str(text)
    
    # lowercase
    text = text.lower()

    # remove punctuation
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # tokenize
    words = text.split()

    # remove stopwords
    words = [w for w in words if w not in stop_words]

    return " ".join(words)


data["clean_resume"] = data["Resume_str"].apply(clean_resume)

data.head()

,Resume_str,Category,clean_resume
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,HR,hr administratormarketing associate hr adminis...
1,"HR SPECIALIST, US HR OPERATIONS ...",HR,hr specialist us hr operations summary versati...
2,HR DIRECTOR Summary Over 2...,HR,hr director summary years experience recruitin...
3,HR SPECIALIST Summary Dedica...,HR,hr specialist summary dedicated driven dynamic...
4,HR MANAGER Skill Highlights ...,HR,hr manager skill highlights hr skills hr depar...


In [5]:
job_description = """
Looking for a data scientist with skills in Python,
machine learning, deep learning, statistics,
data analysis, SQL, and data visualization.
"""

In [6]:
vectorizer = TfidfVectorizer(max_features=5000)

resume_vectors = vectorizer.fit_transform(data["clean_resume"])

job_vector = vectorizer.transform([job_description])

In [7]:
similarity_scores = cosine_similarity(resume_vectors, job_vector)

data["score"] = similarity_scores

data.head()

,Resume_str,Category,clean_resume,score
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,HR,hr administratormarketing associate hr adminis...,0.057748
1,"HR SPECIALIST, US HR OPERATIONS ...",HR,hr specialist us hr operations summary versati...,0.000744
2,HR DIRECTOR Summary Over 2...,HR,hr director summary years experience recruitin...,0.001206
3,HR SPECIALIST Summary Dedica...,HR,hr specialist summary dedicated driven dynamic...,0.003537
4,HR MANAGER Skill Highlights ...,HR,hr manager skill highlights hr skills hr depar...,0.006427


In [8]:
ranked_candidates = data.sort_values(by="score", ascending=False)

ranked_candidates[['Category','score']].head(10)

,Category,score
1218,CONSULTANT,0.390008
1762,ENGINEERING,0.353725
1339,AUTOMOBILE,0.272622
926,AGRICULTURE,0.236247
2153,BANKING,0.221114
1303,DIGITAL-MEDIA,0.210711
331,INFORMATION-TECHNOLOGY,0.207173
1040,SALES,0.206458
929,AGRICULTURE,0.199906
1091,SALES,0.198431


In [9]:
skills_list = [
'python','machine learning','deep learning','sql',
'data analysis','statistics','tensorflow','nlp'
]

In [10]:
def extract_skills(resume):

    resume = resume.lower()

    found_skills = []

    for skill in skills_list:
        if skill in resume:
            found_skills.append(skill)

    return found_skills


data["skills"] = data["clean_resume"].apply(extract_skills)

data.head()

,Resume_str,Category,clean_resume,score,skills
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,HR,hr administratormarketing associate hr adminis...,0.057748,"[data analysis, statistics]"
1,"HR SPECIALIST, US HR OPERATIONS ...",HR,hr specialist us hr operations summary versati...,0.000744,[]
2,HR DIRECTOR Summary Over 2...,HR,hr director summary years experience recruitin...,0.001206,[]
3,HR SPECIALIST Summary Dedica...,HR,hr specialist summary dedicated driven dynamic...,0.003537,[]
4,HR MANAGER Skill Highlights ...,HR,hr manager skill highlights hr skills hr depar...,0.006427,[]


In [11]:
job_skills = set(skills_list)

def skill_gap(candidate_skills):

    candidate_skills = set(candidate_skills)

    missing = job_skills - candidate_skills

    return list(missing)


data["missing_skills"] = data["skills"].apply(skill_gap)

data.head()

,Resume_str,Category,clean_resume,score,skills,missing_skills
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,HR,hr administratormarketing associate hr adminis...,0.057748,"[data analysis, statistics]","[nlp, deep learning, machine learning, tensorf..."
1,"HR SPECIALIST, US HR OPERATIONS ...",HR,hr specialist us hr operations summary versati...,0.000744,[],"[nlp, statistics, deep learning, machine learn..."
2,HR DIRECTOR Summary Over 2...,HR,hr director summary years experience recruitin...,0.001206,[],"[nlp, statistics, deep learning, machine learn..."
3,HR SPECIALIST Summary Dedica...,HR,hr specialist summary dedicated driven dynamic...,0.003537,[],"[nlp, statistics, deep learning, machine learn..."
4,HR MANAGER Skill Highlights ...,HR,hr manager skill highlights hr skills hr depar...,0.006427,[],"[nlp, statistics, deep learning, machine learn..."
